# Loading Data from Github

In [2]:
import pandas as pd
import numpy as np
import ast

In [328]:
df = pd.read_csv(r'D:\ED\TrashPL\data\sma_male_csv2\all_matches.csv')

C:\Users\rajat\AppData\Local\Temp\ipykernel_15532\1119820694.py:1: DtypeWarning: Columns (20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'D:\ED\TrashPL\data\sma_male_csv2\all_matches.csv')


In [336]:
people = pd.read_csv(r'D:\ED\FIFS_IPLungit\data\register_data\people.csv')

# Cleaning data

add the league name, the file will be saved specifically according to this name

In [338]:
league = 'SMAT'

renaming columns

In [339]:
df =  df.rename(columns={'striker':'batter'})

Dropping unnecessary columns

In [340]:

df.drop(columns = ['other_wicket_type', 'other_player_dismissed', 'penalty'], inplace=True)

Fill null values

In [341]:
df['wides'] = df['wides'].fillna(0)
df['noballs'] = df['noballs'].fillna(0)
df['byes'] = df['byes'].fillna(0)
df['legbyes'] = df['legbyes'].fillna(0)

Create new columns

In [342]:
df['batter_balls_faced'] = np.where(df['wides']==0, 1, 0)
df['bowler_runs'] = df['runs_off_bat'] + df['wides'] + df['noballs']

Resolving player name for consistency

In [343]:
people = people[['unique_name', 'key_cricinfo']]

In [344]:
# Create a dictionary mapping unique_name to key_cricinfo
name_to_id = dict(zip(people['unique_name'], people['key_cricinfo']))
# Map batter and bowler names to their respective IDs using the dictionary
df['batter_id'] = df['batter'].map(name_to_id)
df['bowler_id'] = df['bowler'].map(name_to_id)

In [303]:
df[df.match_id == 1162641]

,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,batter,non_striker,...,wides,noballs,byes,legbyes,wicket_type,player_dismissed,batter_balls_faced,bowler_runs,batter_id,bowler_id


In [345]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 474610 entries, 0 to 474609
Data columns (total 23 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   match_id            474610 non-null  int64  
 1   season              474610 non-null  object 
 2   start_date          474610 non-null  object 
 3   venue               474610 non-null  object 
 4   innings             474610 non-null  int64  
 5   ball                474610 non-null  float64
 6   batting_team        474610 non-null  object 
 7   bowling_team        474610 non-null  object 
 8   batter              474610 non-null  object 
 9   non_striker         474610 non-null  object 
 10  bowler              474610 non-null  object 
 11  runs_off_bat        474610 non-null  int64  
 12  extras              474610 non-null  int64  
 13  wides               474610 non-null  float64
 14  noballs             474610 non-null  float64
 15  byes                474610 non-nul

# Batting Scorecard

In [346]:
df.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'batter', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'wicket_type', 'player_dismissed', 'batter_balls_faced', 'bowler_runs',
       'batter_id', 'bowler_id'],
      dtype='object')

In [347]:
df['batting_team'] = df['batting_team'].replace({'Himachal':'Himachal Pradesh'})
df['bowling_team'] = df['bowling_team'].replace({'Himachal':'Himachal Pradesh'})

In [348]:
def process_batting_scorecard_from_kaggle(df):
    if df.empty:
        return pd.DataFrame()

    batting_stats = {}

    for _, ball in df.iterrows():
        match_id = ball['match_id']
        match_date = ball['start_date']
        venue = ball['venue']

        striker = ball['batter_id']
        batsman_name = ball['batter']
        bowler_name = ball['bowler']
        batting_team = ball['batting_team']
        bowling_team = ball['bowling_team']

        key = (match_id, batting_team, striker)
        if key not in batting_stats:
            batting_stats[key] = {
                'batsman_name': batsman_name,
                'bowler_name': "",
                'runs': 0,
                'balls': 0,
                'fours': 0,
                'sixes': 0,
                'isOut': False,
                'wicketType': 'not out',
                'innings': ball['innings'],
                'p_bat_id': striker,
                'p_bowl_id': None,
                'bat_team': batting_team,
                'bowl_team': bowling_team,
                'date': match_date,
                'venue': venue
            }

        stats = batting_stats[key]

        runs = ball['runs_off_bat']
        stats['runs'] += runs

        # Count boundaries
        if runs == 4:
            stats['fours'] += 1
        elif runs == 6:
            stats['sixes'] += 1

        # Count balls faced (excluding wides and no balls)
        if ball['batter_balls_faced'] == 1:
            stats['balls'] += 1

        # Check if batsman is out
        if pd.notna(ball['wicket_type']) and ball['player_dismissed'] == batsman_name:
            stats['isOut'] = True
            stats['wicketType'] = ball['wicket_type']
            stats['bowler_name'] = bowler_name
            stats['p_bowl_id'] = ball['bowler_id']

        batting_stats[key] = stats

    results = []
    for (match_id, team, player), stats in batting_stats.items():
        strike_rate = (stats['runs'] / stats['balls'] * 100) if stats['balls'] > 0 else 0
        results.append({
            'match_id': match_id,
            'date': stats['date'],
            'venue': stats['venue'],
            'inns': stats['innings'],
            'batter_id': player,
            'batter': stats['batsman_name'],
            'bowler': stats['bowler_name'],
            'bowler_id': stats['p_bowl_id'],
            'runs': stats['runs'],
            'balls': stats['balls'],
            'fours': stats['fours'],
            'sixes': stats['sixes'],
            'sr': round(strike_rate, 2),
            'isOut': stats['isOut'],
            'wicketType': stats['wicketType'],
            'bat_team': stats['bat_team'],
            'bowl_team': stats['bowl_team']
        })

    return pd.DataFrame(results)

batting_card=process_batting_scorecard_from_kaggle(df)
batting_card.sort_values(by=['date', 'match_id', 'inns'], inplace=True)
batting_card['bowler'] = batting_card['bowler'].replace("", pd.NA)

In [349]:
batting_card.bat_team.unique()

array(['Gujarat', 'Himachal Pradesh', 'Bengal', 'Hyderabad (India)',
       'Haryana', 'Tamil Nadu', 'Punjab', 'Rajasthan', 'Jammu & Kashmir',
       'Kerala', 'Saurashtra', 'Tripura', 'Andhra', 'Madhya Pradesh',
       'Railways', 'Delhi', 'Odisha', 'Mumbai', 'Karnataka', 'Services',
       'Maharashtra', 'Uttar Pradesh', 'Vidarbha', 'Jharkhand', 'Goa',
       'Baroda', 'Assam', 'Chhattisgarh', 'Manipur', 'Nagaland',
       'Meghalaya', 'Sikkim', 'Arunachal Pradesh', 'Mizoram',
       'Puducherry', 'Uttarakhand', 'Bihar', 'Chandigarh'], dtype=object)

In [350]:
batting_card.head(15)

,match_id,date,venue,inns,batter_id,batter,bowler,bowler_id,runs,balls,fours,sixes,sr,isOut,wicketType,bat_team,bowl_team
2,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",1,340062.0,Priyank Panchal,Nikhil Gangta,528332.0,96,87,6,0,110.34,True,caught,Gujarat,Himachal Pradesh
3,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",1,32242.0,PA Patel,AN Chauhan,537122.0,105,78,15,0,134.62,True,bowled,Gujarat,Himachal Pradesh
4,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",1,390480.0,MC Juneja,Rahul Khajan Singh,500137.0,24,30,0,0,80.00,True,stumped,Gujarat,Himachal Pradesh
6,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",1,290720.0,RH Bhatt,Pankaj Jaswal,652359.0,120,87,9,3,137.93,True,bowled,Gujarat,Himachal Pradesh
7,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",1,604622.0,Himalaya Barad,<NA>,NaN,6,9,0,0,66.67,False,not out,Gujarat,Himachal Pradesh
9,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",1,390545.0,Rohit Dahiya,Pankaj Jaswal,652359.0,21,24,0,0,87.50,True,run out,Gujarat,Himachal Pradesh
10,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",1,496219.0,Chirag Gandhi,AN Chauhan,537122.0,54,45,3,0,120.00,True,caught and bowled,Gujarat,Himachal Pradesh
0,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",2,262464.0,Robin Bist,Roosh Kalaria,528055.0,36,48,6,0,75.00,True,bowled,Himachal Pradesh,Gujarat
1,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",2,500135.0,P Chopra,<NA>,NaN,267,186,21,6,143.55,False,not out,Himachal Pradesh,Gujarat
5,902087,2016-01-02,"Vidarbha Cricket Association Stadium, Jamtha",2,28160.0,P Dogra,HP Patel,793915.0,108,69,9,3,156.52,True,caught,Himachal Pradesh,Gujarat


In [351]:
# Only convert non-null values to int
batting_card.loc[batting_card['batter_id'].notna(), 'batter_id'] = batting_card.loc[batting_card['batter_id'].notna(), 'batter_id'].astype(int)

In [352]:
batting_card

,match_id,date,venue,inns,batter_id,batter,bowler,bowler_id,runs,balls,fours,sixes,sr,isOut,wicketType,bat_team,bowl_team


In [353]:
# Add league column to batting card
batting_card['league'] = league
batting_card.to_csv(f'D:\\ED\\FIFS_IPLungit\\data\\mbm\\cricsheet_{league.lower()}_mbm_bat.csv', index=False)

In [354]:
batting_card.bat_team.unique()

array(['Gujarat', 'Himachal Pradesh', 'Bengal', 'Hyderabad (India)',
       'Haryana', 'Tamil Nadu', 'Punjab', 'Rajasthan', 'Jammu & Kashmir',
       'Kerala', 'Saurashtra', 'Tripura', 'Andhra', 'Madhya Pradesh',
       'Railways', 'Delhi', 'Odisha', 'Mumbai', 'Karnataka', 'Services',
       'Maharashtra', 'Uttar Pradesh', 'Vidarbha', 'Jharkhand', 'Goa',
       'Baroda', 'Assam', 'Chhattisgarh', 'Manipur', 'Nagaland',
       'Meghalaya', 'Sikkim', 'Arunachal Pradesh', 'Mizoram',
       'Puducherry', 'Uttarakhand', 'Bihar', 'Chandigarh'], dtype=object)

# Bowling Card

In [355]:
df.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'batter', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'wicket_type', 'player_dismissed', 'batter_balls_faced', 'bowler_runs',
       'batter_id', 'bowler_id'],
      dtype='object')

In [356]:
df['bowlruns'] = df['runs_off_bat'] + df['wides'] + df['noballs']

In [357]:
df['wicket_type'].unique()

array([nan, 'bowled', 'caught', 'stumped', 'run out', 'caught and bowled',
       'lbw', 'retired hurt', 'hit wicket', 'retired out'], dtype=object)

In [358]:
import pandas as pd
import numpy as np

def calculate_bowling_card(df):
    # Ensure necessary columns are of the right type
    df = df.copy()

    # Convert columns to appropriate types
    df['noballs'] = df['noballs'].fillna(0).astype(float)
    df['wides'] = df['wides'].fillna(0).astype(float)
    df['wicket_type'] = df['wicket_type'].fillna('')
    df['runs_off_bat'] = df['runs_off_bat'].fillna(0).astype(float)
    df['bowler_runs'] = df['bowler_runs'].fillna(0).astype(float)

    # Ensure ball column is string
    df['ball'] = df['ball'].astype(str)

    # Create legal delivery indicator
    df['legal_delivery'] = (df['noballs'] == 0) & (df['wides'] == 0)

    # Create valid wicket indicator (excluding run outs)
    df['valid_wicket'] = (df['wicket_type'].notna()) & (df['wicket_type'] != '') & (df['wicket_type'] != 'run out')

    # Group by match, bowler, bowler ID, innings
    bowling_card = df.groupby(['match_id', 'bowler', 'bowler_id', 'bowling_team', 'batting_team', 'innings'])

    # Compute required statistics
    summary = bowling_card.agg(
        overs=('legal_delivery', lambda x: f"{x.sum() // 6}.{x.sum() % 6}"),  # Overs in x.y format
        balls=('ball', 'count'),  # Total deliveries (including wides and no-balls)
        wickets=('valid_wicket', 'sum'),  # Count of valid wickets (excluding run outs)
        runs=('bowler_runs', 'sum'),  # Total runs conceded
        lbw_or_bowled=('wicket_type', lambda x: ((x == 'lbw') | (x == 'bowled')).sum()),
        dot_balls=('runs_off_bat', lambda x: (x == 0).sum())  # Dot balls where no runs were scored
    ).reset_index()

    # Calculate economy rate
    def calculate_economy(row):
        # Handle cases with 0 overs
        if row['overs'] == '0.0':
            return 0.0  # Or np.inf, depending on how you want to handle this

        # Calculate total overs as a float
        overs_parts = row['overs'].split('.')
        total_overs = int(overs_parts[0]) + int(overs_parts[1]) / 6

        # Prevent division by zero
        return row['runs'] / total_overs if total_overs > 0 else 0.0

    summary['economy'] = summary.apply(calculate_economy, axis=1)

    # Calculate maidens
    maiden_overs = df.groupby(['match_id', 'bowler', 'bowler_id', 'innings', df['ball'].str.split('.').str[0].astype(int)]).agg(
        over_runs=('runs_off_bat', 'sum')
    ).reset_index()

    maidens = maiden_overs.groupby(['match_id', 'bowler', 'bowler_id', 'innings'])['over_runs'].apply(lambda x: (x == 0).sum()).reset_index()
    summary = summary.merge(maidens, on=['match_id', 'bowler', 'bowler_id', 'innings'], how='left')
    summary.rename(columns={'over_runs': 'maidens'}, inplace=True)

    # Add venue and date
    match_info = df.groupby('match_id').agg(
        venue=('venue', 'first'),
        date=('start_date', 'first')
    ).reset_index()
    summary = summary.merge(match_info, on='match_id', how='left')

    # Determine the first over bowled by each player in each innings
    first_over = df[df['legal_delivery']].groupby(['match_id', 'bowler', 'bowler_id', 'innings'])['ball'].apply(
        lambda x: min(int(str(val).split('.')[0]) for val in x)
    ).reset_index()

    summary = summary.merge(first_over, on=['match_id', 'bowler', 'bowler_id', 'innings'], how='left')

    # Sort by match, innings, and first over
    summary = summary.sort_values(by=['match_id', 'innings', 'ball'])

    summary.drop(columns=['ball'], inplace=True)

    return summary

# Example usage
bowling_card = calculate_bowling_card(df)

In [359]:
bowling_card

,match_id,bowler,bowler_id,bowling_team,batting_team,innings,overs,balls,wickets,runs,lbw_or_bowled,dot_balls,economy,maidens,venue,date
7,902087,RG More,424219.0,Himachal Pradesh,Gujarat,1,9.0,60,0,102.0,0,27,11.333333,0,"Vidarbha Cricket Association Stadium, Jamtha",2016-01-02
0,902087,AN Chauhan,537122.0,Himachal Pradesh,Gujarat,1,12.0,78,6,96.0,3,30,8.000000,0,"Vidarbha Cricket Association Stadium, Jamtha",2016-01-02
5,902087,Pankaj Jaswal,652359.0,Himachal Pradesh,Gujarat,1,12.0,72,3,57.0,3,30,4.750000,0,"Vidarbha Cricket Association Stadium, Jamtha",2016-01-02
1,902087,Bipul Sharma,35928.0,Himachal Pradesh,Gujarat,1,9.0,54,0,78.0,0,3,8.666667,0,"Vidarbha Cricket Association Stadium, Jamtha",2016-01-02
9,902087,Rahul Khajan Singh,500137.0,Himachal Pradesh,Gujarat,1,12.0,72,3,75.0,0,12,6.250000,0,"Vidarbha Cricket Association Stadium, Jamtha",2016-01-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8283,1446107,Tripuresh Singh,1460345.0,Madhya Pradesh,Mumbai,2,5.4,34,4,68.0,0,12,12.000000,0,"M Chinnaswamy Stadium, Bengaluru",2024-12-15
8280,1446107,Shivam Shukla,1460342.0,Madhya Pradesh,Mumbai,2,6.0,38,2,44.0,0,14,7.333333,0,"M Chinnaswamy Stadium, Bengaluru",2024-12-15
8276,1446107,Rahul Batham,942649.0,Madhya Pradesh,Mumbai,2,6.0,36,0,68.0,0,10,11.333333,0,"M Chinnaswamy Stadium, Bengaluru",2024-12-15
8284,1446107,VR Iyer,851403.0,Madhya Pradesh,Mumbai,2,6.0,42,2,58.0,0,20,9.666667,0,"M Chinnaswamy Stadium, Bengaluru",2024-12-15


In [360]:
bowling_card[bowling_card['match_id'] == 551048]

,match_id,bowler,bowler_id,bowling_team,batting_team,innings,overs,balls,wickets,runs,lbw_or_bowled,dot_balls,economy,maidens,venue,date


In [361]:
bowling_card.rename(columns={'batting_team':'bat_team', 'bowling_team':'bowl_team', 'innings':'inns', 'economy':'eco', 'dot_balls':'dots'}, inplace=True)

In [362]:
bowling_card.columns

Index(['match_id', 'bowler', 'bowler_id', 'bowl_team', 'bat_team', 'inns',
       'overs', 'balls', 'wickets', 'runs', 'lbw_or_bowled', 'dots', 'eco',
       'maidens', 'venue', 'date'],
      dtype='object')

In [363]:
# Add league column to bowling card
bowling_card['league'] = league
bowling_card.to_csv(f'D:\\ED\\FIFS_IPLungit\\data\\mbm\\cricsheet_{league.lower()}_mbm_bowl.csv', index=False)